In [1]:
import pandas as pd
import numpy as np

In [ ]:
total = pd.Series(dtype = "int64")  # 빈 시리즈 생성 : 1차원 데이터 저장
n_chunks = 0
for chunk in pd.read_csv("../data/web_logs.csv"
                         , usecols=["event_type"]   # event_type 컬럼 추출
                         , chunksize=200000):       # 20만 개씩 잘라 chunk로 보냄.

    total= total.add(chunk["event_type"].value_counts(), fill_value=0)  # add는 누적(처음은 20만 건, 두 번째는 40만 건?)
    n_chunks += 1                                                       # 기존 Series에서 add Series를 더함. cart끼리, purchase끼리 더함.
                                                                        # 자르고 어떻게 처리하는지가 핵심.
print(f"처리한 청크 수: {n_chunks}")
print(total)

처리한 청크 수: 5
event_type
cart        119434.0
purchase     80359.0
search      200206.0
view        600001.0
dtype: float64


In [9]:
full_vc = pd.read_csv("../data/web_logs.csv")["event_type"]
full_vc.value_counts()

event_type
view        600001
search      200206
cart        119434
purchase     80359
Name: count, dtype: int64

In [ ]:
# Parquet / CSV 비교: 크기, 속도
logs = pd.read_csv("../data/web_logs.csv")
logs["event_type"]=logs["event_type"].astype("category")

# Parquet 변환
logs.to_parquet("../data/web_logs.parquet", engine="pyarrow")

In [13]:
import time, os
# 읽기 속도 비교
# csv 파일 읽고 시간 계산
t0 = time.perf_counter()    # 시작 시간 저장
_ = pd.read_csv("../data/web_logs.csv")
t1 = time.perf_counter() # 끝난 시간 저장
t_csv = t1 - t0

# parquet 파일 읽는 시간 계산
t0 = time.perf_counter()
_ = pd.read_parquet("../data/web_logs.parquet", engine="pyarrow")
t1 = time.perf_counter()
t_pq = t1 - t0

print(f"CSV read time : {t_csv}, PQ read time: {t_pq}")

CSV read time : 1.644986099912785, PQ read time: 0.06445109995547682
